In [15]:
%load_ext autoreload
%autoreload 2

## Initial Test

### Draw

Make Chip

In [ ]:
from quantrolib.chip import JAWS

chip = JAWS()

In [3]:
chip.draw()

Place Resonators

In [ ]:
3189 - 680/2 - 25

In [5]:
from quantrolib.resonator import IncaResonatorShortedMasked

resonator = IncaResonatorShortedMasked(
    design=chip,
    options=dict(center_offset=1),
)
resonator = IncaResonatorShortedMasked(
    design=chip,
    options=dict(center_offset=0),
)
resonator = IncaResonatorShortedMasked(
    design=chip,
    options=dict(center_offset=-1),
)

# resonator = Resonator(
#     design=chip,
#     options=dict(pos_x="2824um", pos_y="0um", orientation="0"),
# )


In [ ]:
chip.draw()

Add DC Probes

In [78]:
from qiskit_metal.qlibrary.tlines.framed_path import RouteFramed


DC_probe_1 = RouteFramed(chip._design, 'DC_probe_1', options=dict(
    pin_inputs=dict(
        start_pin=dict(component='rf_port_2', pin='out'),
        end_pin=dict(component='resonator_1', pin='DC_probe'),
    ),
    lead=dict(end_straight='2mm', start_straight='0.4mm'),
))

# DC_probe_2 = RouteFramed(chip._design, 'DC_probe_2', options=dict(
#     pin_inputs=dict(
#         start_pin=dict(component='rf_port_3', pin='rf_port_3'),
#         end_pin=dict(component='resonator_2', pin='DC_probe'),
#     ),
#     lead=dict(end_straight='2mm', start_straight='0.4mm'),
# ))


In [79]:
chip.draw()

In [9]:
transmission_1 = RouteFramed(chip._design, 'transmission_1', options=dict(
    pin_inputs=dict(
        start_pin=dict(component='rf_port_0', pin='out'),
        end_pin=dict(component='rf_port_1', pin='out'),
    ),
))

# transmission_2 = RouteFramed(chip._design, 'transmission_2', options=dict(
#     pin_inputs=dict(
#         start_pin=dict(component='rf_port_5', pin='rf_port_5'),
#         end_pin=dict(component='rf_port_4', pin='rf_port_4'),
#     ),
# ))

In [10]:
chip.draw()

Generate GDS

In [ ]:
chip.generate_gds("test_chip")


### Eigenmode Simulation

In [6]:
design = chip._design

#### Edit params of specific Components

In [7]:
import numpy as np

wire_width = 2  # um
wire_length = 94  # um
real_wire_width = 0.6 # um

kinetic_inductance_square = 2.e-13  # H/square
nanowire_inductance = kinetic_inductance_square * wire_width/real_wire_width * wire_length

design.components["resonator_1"].options.hfss_inductance = f"{nanowire_inductance*1e12} pH"


In [ ]:
design.components["resonator_1"].options

In [9]:
design.rebuild()

#### Simulation Setup

In [10]:
from qiskit_metal.renderers.renderer_ansys.hfss_renderer import QHFSSRenderer
fourq_hfss : QHFSSRenderer = design.renderers.hfss

In [ ]:
fourq_hfss.start()

In [ ]:
fourq_hfss.connect_ansys_design()

In [ ]:
fourq_hfss.new_ansys_design("HFSSMetalEigenmode", 'eigenmode')

fourq_hfss.connect_ansys_design("HFSSMetalEigenmode")

Initialize

All components in the design:

In [ ]:
tuple(chip._design.components)

In [13]:
port_list = [(f'rf_port_{i}', 'in', 50) for i in range(6)]

In [ ]:
port_list

#### Run the Simulation

In [ ]:
fourq_hfss.options["max_mesh_length_jj"] = "0.5um"
fourq_hfss.options["max_mesh_length_port"] = "250um"
fourq_hfss.options

Rebuild to verfiy changes?

In [16]:
fourq_hfss.clean_active_design()
fourq_hfss.render_design(
    open_pins=[],
    port_list=port_list,
)

In [ ]:
fourq_hfss.initialize_eigenmode(
    name='Setup',
    min_freq_ghz=4,
    n_modes=3,
    max_delta_f=0.1,
    max_passes=10,
)

In [ ]:
fourq_hfss.activate_ansys_setup('Setup')

In [ ]:
fourq_hfss.analyze_setup('Setup')

In [ ]:
create_report = fourq_hfss.plot_fields(
    object_name="main",
    QuantityName='Mag_E',
)

In [ ]:
type(create_report)

In [ ]:
fourq_hfss.get_unique_component_ids()

## Framework

In [1]:
from quantrolib.simulation import ANSYS, RenderConfig, SimulationConfig, ReportConfig


### Configs

## Simple Chiplet

### Design

In [2]:
from quantrolib.chip import Chiplet
from quantrolib.resonator import IncaResonatorShortedMasked, Resonator

def build_chiplet(resonator_type: str) -> Chiplet:
    chip = Chiplet(size_x="2mm", size_y="2mm")

    if resonator_type == "INCA":
        resonator = IncaResonatorShortedMasked(
            name="resonator",
            design=chip,
            options=dict(center_offset=-1),
        )
    
    elif resonator_type == "NEW":

        resonator = Resonator(
            name="resonator",
            design=chip,
            options=dict(pos_x="2824um", pos_y="0um", orientation="0"),
        )
    else:
        raise ValueError(f"Unknown resonator type '{resonator_type}'.")

    return chip

### Draw

In [3]:
chip = build_chiplet("INCA")

In [4]:
chip.draw()

In [ ]:
chip.generate_gds("test_chip")

### Post-process design

In [6]:
design = chip._design

In [ ]:
tuple(design.components)

In [8]:
import numpy as np

wire_width = 2  # um
wire_length = 94  # um
real_wire_width = 0.6 # um

kinetic_inductance_square = 2.e-13  # H/square
nanowire_inductance = kinetic_inductance_square * wire_width/real_wire_width * wire_length

design.components["resonator"].options.hfss_inductance = f"{nanowire_inductance*1e12} pH"

### Simulate

In [9]:
design.rebuild()

In [ ]:
render_config = RenderConfig(
    name="Render",
    design=design,
    # project_path="C:/Users/rober/OneDrive/ETH/QuantumComputing/QuantroLib/quantrolib/examples",
    # project_name="HFSSMetalEigenmode",
    # design_name="HFSSMetalEigenmode_bis",
    open_pins=[],
    port_list=[],
    max_mesh_length_jj="0.5um",
    max_mesh_length_port="250um",
)

simulation_config = SimulationConfig(
    name="Setup",
    min_freq_ghz=4.0,
    n_modes=3,
    max_delta_f=0.1,
    max_passes=10,
)

report_config = ReportConfig(
    name="Fields",
    field_configs=[{"object_name": "main", "QuantityName": 'Mag_E', 'PlotFolder': 'E Field'}, {"object_name": "main", "QuantityName": 'Mag_H', 'PlotFolder': 'H Field'}]
)

ansys = ANSYS(configs=[render_config, simulation_config, report_config], run_upon_init=True)

In [ ]:
report_config = ReportConfig(
    name="Fields",
    field_configs=[{"object_name": "main", "QuantityName": 'Mag_E'}, {"object_name": "main", "QuantityName": 'Mag_H', 'PlotFolder': 'H Field'}]
)
ansys.add_config(report_config)
ansys.run_report()
